# Join the DistribAI grid from Colab (free GPU)

Turn this free GPU session into a grid worker. The operator runs one command to
publish the grid (a free Cloudflare quick tunnel, no account and no domain) and
sends you a join link. Paste that link below and run the notebook.

The worker is the native C++ binary: it registers over HTTP, claims one training
replica at a time, runs the LibTorch trainer as a limited child, and reports a
gradient envelope back to the coordinator. Sessions are burst workers: they join
while this runtime is alive and leave when it ends.

Host note: Google Colab allows outbound HTTPS, which is why tunnel links work
here even when a raw port would be filtered.

In [ ]:
# Paste the join link (or a bare host:port) the operator sent you.
JOIN_LINK = "distribai://join?v=1&engine=native&provider=cloudflare&host=REPLACE.trycloudflare.com&port=443&tls=1&invite=team-alpha"
NODE_ID = ""  # optional: a stable name for this node

# Kept to the standard library so the link can be read before anything is installed.
import os
from urllib.parse import parse_qs, urlsplit


def parse_join_link(text):
    """Turn a join link into the env the worker reads."""
    text = text.strip()
    if text.startswith("distribai://"):
        q = parse_qs(urlsplit(text).query)
        host = q["host"][0]
        port = q.get("port", ["50061"])[0]
        scheme = "https" if q.get("tls", ["0"])[0] == "1" else "http"
        out = {"ORCHESTRATOR_URL": f"{scheme}://{host}:{port}"}
        if "invite" in q:
            out["DISTRIBAI_INVITE_CODE"] = q["invite"][0]
        return out
    if text.startswith("http"):
        return {"ORCHESTRATOR_URL": text.rstrip("/")}
    return {"ORCHESTRATOR_URL": f"http://{text}"}


ENV = parse_join_link(JOIN_LINK)
ENV["DISTRIBAI_EPHEMERAL"] = "1"  # burst session: nothing long-lived on disk
print("joining", ENV["ORCHESTRATOR_URL"])
print("invite:", "set" if "DISTRIBAI_INVITE_CODE" in ENV else "not needed")

In [ ]:
WORKDIR = "/content/DistribAI"
REPO = "https://github.com/naxium-oss/DistribAI.git"
import subprocess

# The worker is C++. Clone the repo and build the grid binaries.
subprocess.run(["git", "clone", "--depth", "1", REPO, WORKDIR], check=False)
subprocess.run(["make", "grid"], cwd=WORKDIR, check=True)
# LibTorch trainer: this host already ships torch, so the trainer builds here too.
subprocess.run(["make", "torch"], cwd=WORKDIR, check=False)

In [ ]:
import os
import subprocess

WORK_DIR = f"{WORKDIR}/runtime/grid/worker"
argv = [f"{WORKDIR}/build/cpp_port/distribai_worker", "--orchestrator", ENV["ORCHESTRATOR_URL"]]
if ENV.get("DISTRIBAI_INVITE_CODE"):
    argv += ["--invite", ENV["DISTRIBAI_INVITE_CODE"]]
argv += ["--work-dir", WORK_DIR]
if NODE_ID:
    argv += ["--node-id", NODE_ID]

print(" ".join(argv))
# Ctrl-C, or the end of the session, disconnects this node.
subprocess.run(argv, env={**os.environ, **ENV})

## Notes

- Free GPU sessions are burst workers: the node drops when the runtime ends, and
  re-running this notebook rejoins it.
- The tunnel link uses port 443 with TLS, so hosts that block odd ports still work.
- Jobs sent here run the LibTorch trainer for the job you claim, under rlimits set
  by the worker (`--mem-mb`, `--cpu-sec`).
- Prefer short job slices; keep checkpoints on storage the operator can read.
- Nothing here needs a paid plan or a custom domain.
